In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### Fix problem

In [8]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl
df = df[
    df["ISSUE"].str.contains("Missing", case=False, na=False)
]


len(df)

df =  df[['ISSUE','Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)
df["Network ID"] = pd.to_numeric(df["Network ID"], errors="coerce").astype("Int64")
df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")
df


,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,Unique Location (String)
0,"Attribution, ID, Location, Missing Data",2628,9,BC-Air -> MVRD-AIR,M110514 -> T038,Annacis Island,"122.9422222 W, 49.15972222 N, Elev. 160 m -> 1..."
1,"Attribution, ID, Location, Missing Data (Diffe...",12220,9,BC-Air -> MVRD-AIR,E232244 -> T020,Pitt Meadows Meadowlands School,"122.7089 W, 49.2453 N, Elev. 20 m -> 122.7089 ..."
2,"Attribution, ID, Missing Data",12092,9,BC-Air -> MVRD-AIR,E207418 -> T018,Burnaby South,"122.9856 W, 49.2153 N, Elev. 122 m"
3,"Attribution, ID, Missing Data",2634,9,BC-Air -> MVRD-AIR,M104271 -> T035,Horseshoe Bay Met,"123.2797222 W, 49.37277778 N, Elev. 60 m -> 12..."
4,"Attribution, ID, Missing Data",12269,9,BC-Air -> MVRD-AIR,E207417 -> T17,Richmond South,"123.1083 W, 49.1414 N, Elev. 15 m"
5,"Attribution, ID, Name, Location, missing data",12530,9,BC-Air -> MVRD-AIR,Agassiz Municipal Hall ->T044,NA -> Agassiz Municiple Hall,"121.762334 W, 49.238032 N, Elev. null m -> 121..."
6,"Attribution, ID, Name, Location, Missing Data",12589,9,BC-Air -> MVRD-AIR,Abbotsford Central -> T033,NA -> Abbotsford-Mill Lake,"122.3097 W, 49.0428 N, Elev. null m -> 122.308..."
7,"Attribution, ID, Name, Location, Missing Data",12532,9,BC-Air -> MVRD-AIR,Burnaby Mountain -> T014,NA -> Burnaby Mountain,"122.9222 W, 49.2797 N, Elev. null m -> 122.922..."
8,"Attribution, ID, Name, Location, Missing Data",12593,9,BC-Air -> MVRD-AIR,Burnaby North Eton -> T024,NA -> Burnaby North,"123.0078 W, 49.2875 N, Elev. null m -> 123.008..."
9,"Attribution, ID, Name, Location, Missing Data",12533,9,BC-Air -> MVRD-AIR,Burnaby South -> T018,NA -> Burnaby South,"122.9856 W, 49.2153 N, Elev. null m -> 122.985..."


In [9]:
### Split ID

# Split on '->'
split_ids = df['Native ID'].astype(str).str.split('->', expand=True)

df['old_native_id'] = split_ids[0].str.strip()
df['new_native_id'] = split_ids[1].str.strip()

df = df.drop(columns=['Native ID'])
df["Station ID"] = pd.to_numeric(df["Station ID"], errors="coerce").astype("Int64")

df

,ISSUE,Station ID,Network ID,Network Name,Unique Names,Unique Location (String),old_native_id,new_native_id
0,"Attribution, ID, Location, Missing Data",2628,9,BC-Air -> MVRD-AIR,Annacis Island,"122.9422222 W, 49.15972222 N, Elev. 160 m -> 1...",M110514,T038
1,"Attribution, ID, Location, Missing Data (Diffe...",12220,9,BC-Air -> MVRD-AIR,Pitt Meadows Meadowlands School,"122.7089 W, 49.2453 N, Elev. 20 m -> 122.7089 ...",E232244,T020
2,"Attribution, ID, Missing Data",12092,9,BC-Air -> MVRD-AIR,Burnaby South,"122.9856 W, 49.2153 N, Elev. 122 m",E207418,T018
3,"Attribution, ID, Missing Data",2634,9,BC-Air -> MVRD-AIR,Horseshoe Bay Met,"123.2797222 W, 49.37277778 N, Elev. 60 m -> 12...",M104271,T035
4,"Attribution, ID, Missing Data",12269,9,BC-Air -> MVRD-AIR,Richmond South,"123.1083 W, 49.1414 N, Elev. 15 m",E207417,T17
5,"Attribution, ID, Name, Location, missing data",12530,9,BC-Air -> MVRD-AIR,NA -> Agassiz Municiple Hall,"121.762334 W, 49.238032 N, Elev. null m -> 121...",Agassiz Municipal Hall,T044
6,"Attribution, ID, Name, Location, Missing Data",12589,9,BC-Air -> MVRD-AIR,NA -> Abbotsford-Mill Lake,"122.3097 W, 49.0428 N, Elev. null m -> 122.308...",Abbotsford Central,T033
7,"Attribution, ID, Name, Location, Missing Data",12532,9,BC-Air -> MVRD-AIR,NA -> Burnaby Mountain,"122.9222 W, 49.2797 N, Elev. null m -> 122.922...",Burnaby Mountain,T014
8,"Attribution, ID, Name, Location, Missing Data",12593,9,BC-Air -> MVRD-AIR,NA -> Burnaby North,"123.0078 W, 49.2875 N, Elev. null m -> 123.008...",Burnaby North Eton,T024
9,"Attribution, ID, Name, Location, Missing Data",12533,9,BC-Air -> MVRD-AIR,NA -> Burnaby South,"122.9856 W, 49.2153 N, Elev. null m -> 122.985...",Burnaby South,T018


In [11]:
from sqlalchemy import text
import pandas as pd

query = text("""
SELECT
    COUNT(*)        AS n_obs,
    MIN(o.obs_time) AS oldest_obs_time,
    MAX(o.obs_time) AS newest_obs_time
FROM meta_history m
JOIN obs_raw o
  ON m.history_id = o.history_id
WHERE m.station_id = :station_id
""")

def get_station_obs_stats(station_id, engine):
    df = pd.read_sql(
        query,
        engine,
        params={"station_id": int(station_id)}
    )
    return df.iloc[0]   # returns Series: n_obs, oldest, newest

stats = df["Station ID"].apply(
    lambda sid: get_station_obs_stats(sid, engine)
)

df[["n_obs", "oldest_obs_time", "newest_obs_time"]] = stats

In [13]:
df.to_csv("missing_record_cover.csv", index=False)

In [16]:
#### Split Name

def split_station_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    

df[['old_station_name', 'new_station_name', 'n_names']] = (
    df['Unique Names'].apply(split_station_name)
)

df = df.drop(columns= 'Unique Names')

In [17]:
import re


def _parse_single_location(loc_str):
    """
    Parse:
    '123.764438 W, 48.51616 N, Elev. 512.14 m'
    '123.764438 W, 48.51616 N, Elev. null m'

    Returns: (lat, lon, elev)
    """
    if pd.isna(loc_str):
        return np.nan, np.nan, np.nan

    pattern = (
        r"([\d\.]+)\s*([EW]),\s*"
        r"([\d\.]+)\s*([NS]),\s*"
        r"Elev\.\s*(null|[\d\.]+)\s*m"
    )

    m = re.search(pattern, loc_str, re.IGNORECASE)
    if not m:
        return np.nan, np.nan, np.nan

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    lon = float(lon_val) * (-1 if lon_dir.upper() == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir.upper() == "S" else 1)

    elev = np.nan if elev_val.lower() == "null" else float(elev_val)

    return lat, lon, elev


def parse_old_new_lat_lon_elev(loc_str):
    """
    Returns:
    (old_lat, old_lon, old_elev, new_lat, new_lon, new_elev)
    """
    if pd.isna(loc_str):
        return (np.nan,) * 6

    if "->" in loc_str:
        old_str, new_str = map(str.strip, loc_str.split("->", 1))
    else:
        old_str = new_str = loc_str.strip()

    old_lat, old_lon, old_elev = _parse_single_location(old_str)
    new_lat, new_lon, new_elev = _parse_single_location(new_str)

    return (
        old_lat, old_lon, old_elev,
        new_lat, new_lon, new_elev
    )

cols = [
    'old_lat', 'old_lon', 'old_elev',
    'new_lat', 'new_lon', 'new_elev'
]

df[cols] = df['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_old_new_lat_lon_elev(x))
)

# 
df = df.drop(columns=['Unique Location (String)'])
df


,ISSUE,Station ID,Network ID,Network Name,old_native_id,new_native_id,old_station_name,new_station_name,n_names,old_lat,old_lon,old_elev,new_lat,new_lon,new_elev
0,Missing Data,12203,9,BC-Air -> MVRD-AIR,E308566,T046,New Westminster Sapperton Park,Sapperton Park,2,49.227045,-122.894487,45.0,49.227045,-122.894449,45.0
1,Missing Data,12205,9,BC-Air -> MVRD-AIR,E207723,T013,North Delta,North Delta,1,49.158300,-122.901700,111.0,49.158300,-122.901700,111.0
